# Experiment: Module 11B — Google Colab CUDA prediction parity

**Before running:** select a GPU runtime in Colab. This notebook has no CPU or MPS fallback and has intentionally not been executed outside CUDA.

**Objective.** Run the exact seed-42 adapter and registered synthetic fixture on a real CUDA GPU, record hardware/software evidence, and compare calibrated probabilities and deterministic routing decisions with the committed Mac MPS reference.

This is an execution-parity experiment, not new model training or model-quality evaluation. It never loads the official BANKING77 test split.

In [1]:
from pathlib import Path

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (current, *current.parents) if (path / 'pyproject.toml').is_file()),
    Path('/content/governed-banking-intent-router'),
)
'Notebook location initialised without accessing an accelerator'

'Notebook location initialised without accessing an accelerator'

In [2]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('A real CUDA GPU is required; no CPU or MPS fallback is permitted')
CUDA_DEVICE_INDEX = 0
properties = torch.cuda.get_device_properties(CUDA_DEVICE_INDEX)
{
    'cuda_device': properties.name,
    'cuda_build': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(),
    'memory_bytes': properties.total_memory,
}

RuntimeError: A real CUDA GPU is required; no CPU or MPS fallback is permitted

## Obtain the exact repository revision

The public GitHub repository is checked out at the exact commit containing the completed Module 11 implementation and MPS reference report. A detached checkout prevents branch movement during the run.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/BenjaminAkyen/governed-banking-intent-router.git'
GIT_REVISION = 'd7a931790526454a1ea19b53c36e95c262fa1a62'
PROJECT_ROOT = Path('/content/governed-banking-intent-router')
if PROJECT_ROOT.exists():
    raise RuntimeError('Clean /content before running; an existing checkout is not reused')
subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '--detach', GIT_REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], check=True, text=True, capture_output=True).stdout.strip()

## Restore the private, hash-bound adapter

Model artifacts are deliberately excluded from Git. In Google Drive create `MyDrive/governed-banking-intent-router-private/seed-42/` containing the three original files: `README.md`, `adapter_config.json`, and `adapter_model.safetensors`. The notebook rejects any file whose SHA-256 differs from the Module 10 service registration. No demonstration or mocked weights are accepted.

In [ ]:
from google.colab import drive
import shutil

from governed_banking.data import sha256_file

drive.mount('/content/drive')
private_source = Path('/content/drive/MyDrive/governed-banking-intent-router-private/seed-42')
adapter_destination = PROJECT_ROOT / 'artifacts/multiseed-lora/seed-42'
expected_adapter_hashes = {
    'README.md': '0b5a527d7c8ad39b8025c1fac0fc9865093e3b8f5686193127c7a0f57a8eecde',
    'adapter_config.json': '3d2863d402107e9d0516fcb0e7c4a148cbe2a447777bbca63381d7458cb40bdc',
    'adapter_model.safetensors': 'b78b2dafce23a633c86b962cb672b8b91c8e07c5308debf124a73ffbcb21cca8',
}
for name, expected_hash in expected_adapter_hashes.items():
    source = private_source / name
    if not source.is_file() or sha256_file(source) != expected_hash:
        raise RuntimeError(f'Private adapter file failed hash verification: {name}')
adapter_destination.mkdir(parents=True, exist_ok=False)
for name in expected_adapter_hashes:
    shutil.copy2(private_source / name, adapter_destination / name)
'Exact private adapter restored and verified'

## Resolve the pinned public base model

Network access is used only to download the six registered files from the exact RoBERTa commit. Inference remains offline after the snapshot is populated.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='FacebookAI/roberta-base',
    revision='e2da8e2f811d1448a5b465c236feacd80ffbac7b',
    allow_patterns=[
        'config.json', 'merges.txt', 'model.safetensors',
        'tokenizer.json', 'tokenizer_config.json', 'vocab.json',
    ],
    cache_dir=str(PROJECT_ROOT / 'artifacts/huggingface'),
)
'Pinned RoBERTa snapshot is available for offline inference'

In [ ]:
from governed_banking.runtime_evidence import RuntimeProfile, run_runtime_verification

cuda_profile = RuntimeProfile.from_yaml(PROJECT_ROOT / 'configs/runtime/cuda.yaml')
cuda_runtime_report = run_runtime_verification(
    cuda_profile,
    report_path=PROJECT_ROOT / 'reports/runtime/cuda-runtime.json',
    implementation_paths={
        'accelerator.py': PROJECT_ROOT / 'src/governed_banking/accelerator.py',
        'runtime_evidence.py': PROJECT_ROOT / 'src/governed_banking/runtime_evidence.py',
        'verify_accelerator.py': PROJECT_ROOT / 'scripts/verify_accelerator.py',
    },
    seed=42,
)
{
    'selected': cuda_runtime_report['runtime']['selected'],
    'accelerator': cuda_runtime_report['runtime']['accelerator_name'],
    'report_sha256': cuda_runtime_report['report_sha256'],
}

In [ ]:
from governed_banking.parity import (
    PredictionParityConfig, compare_backend_reports, run_backend_evidence,
)

parity_config = PredictionParityConfig.from_yaml(PROJECT_ROOT / 'configs/prediction_parity.yaml')
cuda_report_path = PROJECT_ROOT / 'reports/parity/cuda-seed42.json'
cuda_report = run_backend_evidence(
    parity_config,
    backend='cuda',
    report_path=cuda_report_path,
    implementation_paths={
        'accelerator.py': PROJECT_ROOT / 'src/governed_banking/accelerator.py',
        'parity.py': PROJECT_ROOT / 'src/governed_banking/parity.py',
        'portable_inference.py': PROJECT_ROOT / 'src/governed_banking/portable_inference.py',
        'run_backend_parity.py': PROJECT_ROOT / 'scripts/run_backend_parity.py',
    },
)
comparison_path = PROJECT_ROOT / 'reports/parity/mps-cuda-comparison.json'
comparison = compare_backend_reports(
    parity_config,
    reference_report_path=PROJECT_ROOT / 'reports/parity/mps-seed42.json',
    candidate_report_path=cuda_report_path,
    comparison_path=comparison_path,
)
comparison['results']

## Decision

The run passes only if all top-1 intents and deterministic routing actions match and the largest absolute probability difference is at most 0.001. A failed gate is evidence to investigate; it must not be hidden by widening the tolerance after observing the result.

In [ ]:
assert comparison['results']['all_gates_passed'], comparison['results']
from google.colab import files

for artifact in (
    PROJECT_ROOT / 'reports/runtime/cuda-runtime.json',
    cuda_report_path,
    comparison_path,
):
    files.download(str(artifact))
'All registered CUDA and cross-device parity gates passed'